# Verify Model Endpoints

This notebook verifies that the LLM and embedding model endpoints configured in `.env` are accessible and working correctly. Models are assumed to be already deployed on RHOAI (or any OpenAI-compatible endpoint).

## 1. Load Environment

In [ ]:
import os
import subprocess

import httpx
from dotenv import load_dotenv
from openai import OpenAI

env_path = os.path.join(os.path.dirname(os.getcwd()), ".env")
load_dotenv(env_path, override=True)

LLM_BASE_URL = os.getenv("LLM_BASE_URL", "")
LLM_API_KEY = os.getenv("LLM_API_KEY", "")
LLM_MODEL = os.getenv("LLM_MODEL", "granite-3.3-8b-instruct")
EMBEDDING_BASE_URL = os.getenv("EMBEDDING_BASE_URL", "")
EMBEDDING_API_KEY = os.getenv("EMBEDDING_API_KEY", "")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "granite-embedding-278m-multilingual")
_VERIFY_SSL = os.getenv("VERIFY_SSL", "true").lower() not in ("false", "0", "no")

# MaaS API key takes priority; fallback to LLM_API_KEY (direct route / OC token)
MAAS_API_KEY = os.getenv("MAAS_API_KEY", "")
if not MAAS_API_KEY and not LLM_API_KEY:
    MAAS_API_KEY = subprocess.run(["oc", "whoami", "-t"], capture_output=True, text=True).stdout.strip()
_effective_llm_key = MAAS_API_KEY if MAAS_API_KEY else LLM_API_KEY

_http_client = None if _VERIFY_SSL else httpx.Client(verify=False, timeout=httpx.Timeout(300.0))

print(f"LLM endpoint:       {LLM_BASE_URL}")
print(f"LLM model:          {LLM_MODEL}")
print(f"LLM auth:           {'MaaS API key' if MAAS_API_KEY else 'LLM_API_KEY'}")
print(f"Embedding endpoint: {EMBEDDING_BASE_URL}")
print(f"Embedding model:    {EMBEDDING_MODEL}")
if not _VERIFY_SSL:
    print(f"SSL verification:   disabled")

if not LLM_BASE_URL or "<your" in LLM_BASE_URL:
    print("\n❌ LLM_BASE_URL not configured. Edit .env with your model endpoint.")
elif not EMBEDDING_BASE_URL or "<your" in EMBEDDING_BASE_URL:
    print("\n❌ EMBEDDING_BASE_URL not configured. Edit .env with your embedding endpoint.")
else:
    print("\n✅ Environment loaded")

## 2. Verify LLM Inference

Send a test prompt to the LLM endpoint.

In [ ]:
llm_client = OpenAI(base_url=LLM_BASE_URL, api_key=_effective_llm_key,
                    http_client=_http_client)

try:
    response = llm_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": "Respond with exactly: Hello from RHOAI"}],
        max_tokens=20,
        temperature=0.0,
    )
    print(f"LLM response: {response.choices[0].message.content}")
    print(f"Model:        {response.model}")
    print(f"Tokens used:  {response.usage.total_tokens}")
    print("\n✅ LLM endpoint working")
except Exception as e:
    print(f"❌ LLM error: {e}")
    print("   Check LLM_BASE_URL and LLM_API_KEY in .env")

## 3. Verify Embedding Model

Generate a test embedding to confirm the model is accessible.

In [ ]:
embedding_client = OpenAI(base_url=EMBEDDING_BASE_URL, api_key=EMBEDDING_API_KEY,
                         http_client=_http_client)

try:
    response = embedding_client.embeddings.create(
        input=["This is a test sentence for embedding verification."],
        model=EMBEDDING_MODEL,
    )
    embedding = response.data[0].embedding
    print(f"Embedding dimension: {len(embedding)}")
    print(f"First 5 values:      {embedding[:5]}")
    print(f"Model:               {response.model}")
    print("\n✅ Embedding endpoint working")
except Exception as e:
    print(f"❌ Embedding error: {e}")
    print("   Check EMBEDDING_BASE_URL and EMBEDDING_API_KEY in .env")

## 4. Verify Batch Embedding

Test batch embedding (used during document ingestion).

In [ ]:
batch_texts = [
    "Document processing with Docling",
    "Multi-agent orchestration via A2A protocol",
    "Semantic search using pgvector",
    "LangGraph stateful agent workflows",
    "Red Hat OpenShift AI model serving",
]

try:
    response = embedding_client.embeddings.create(
        input=batch_texts,
        model=EMBEDDING_MODEL,
    )
    print(f"Batch size:    {len(batch_texts)}")
    print(f"Embeddings:    {len(response.data)}")
    print(f"Dimension:     {len(response.data[0].embedding)}")
    print(f"Total tokens:  {response.usage.total_tokens}")
    print("\n✅ Batch embedding working — ready for document ingestion")
except Exception as e:
    print(f"❌ Batch embedding error: {e}")

## 5. Summary

If all checks passed, you are ready to proceed to Phase 1 (Document Processing).

In [ ]:
print("Model Verification Summary")
print("=" * 40)
print(f"LLM:       {LLM_MODEL}")
print(f"           {LLM_BASE_URL}")
print(f"Embedding: {EMBEDDING_MODEL}")
print(f"           {EMBEDDING_BASE_URL}")
print()
print("Next: Run 1_document_processing/2_document_ingestion.ipynb")
print("✅ Phase 0 complete")